![Kenya Food Price Early Warning System](images/banner.png)

# Kenya Food Price Early Warning System

Forecasting staple food prices across Kenyan markets to give farmers, traders, and food security actors the early signal they currently lack.

## 1. Business Understanding

### 1.1 Background

Kenya's food markets experience recurring price volatility that is structural rather than incidental. Staple commodities such as maize and beans regularly swing between price spikes and crashes across the country's markets, driven by a combination of seasonal harvest cycles, rainfall variability, and fragmented market information. These swings are not isolated shocks; they are a persistent feature of the agricultural economy, and they impose real costs on both ends of the supply chain.

Smallholder farmers frequently sell too early, often under financial pressure, missing the benefit of a later price rise, or they sell too late and absorb the loss of a price crash. On the consumer side, the same volatility worsens food insecurity for urban and rural households already living close to the margin, since a price spike in a staple commodity disproportionately affects the poorest families. County agricultural offices, national drought monitoring bodies, and humanitarian organizations that operate in this space currently rely on lagging, manual, and fragmented indicators, which limits their ability to intervene before a crisis has already taken hold.

This project addresses that information gap directly. Rather than producing a single national average or a static report, it proposes a forecasting and early warning system built at the market and commodity level, so that the people who make real time decisions, buying, selling, storing, or intervening, have a forward looking signal rather than a historical one.

1.7 Data Mining Success Criteria
From a technical standpoint, the modeling work will be considered successful if the forecasting models achieve a MAPE that represents a clear improvement over a naive seasonal baseline across the majority of market and commodity combinations tested. The anomaly detection threshold should be calibrated so that it captures genuine, previously observed historical price shocks in backtesting, without triggering excessively on ordinary short term price noise. The data pipeline should run end to end without manual intervention once source data is refreshed, and the join between price and weather data should retain the large majority of records without significant data loss. Finally, the deployed dashboard should load and update correctly for at least the primary set of markets and commodities selected for the final submission, demonstrating that the system functions as a genuinely usable tool rather than a one time analysis.

1.8 Project Plan Overview
The project follows a five week timeline aligned with the capstone's CRISP-DM structure.

In week one, the team pitches the project, pulls and inspects the HDX price dataset and a sample NASA POWER API call, and confirms that the market name to coordinate join is clean and reliable. In week two, the full data pipeline is built, including cleaning, joining, feature engineering, and the construction of lagged weather features, alongside exploratory data analysis at the commodity and market level. In week three, the Prophet baseline model is built and backtested, the LSTM model is developed in parallel, and the first version of the Streamlit dashboard is deployed. In week four, the model comparison is finalized, the anomaly detection and alert logic is implemented, the CRISP-DM report is written, and feedback from the technical mentor is incorporated. In week five, the dashboard is polished, a WhatsApp alert simulation is added if time permits, and the final demonstration is prepared.

This phased structure ensures that a working, deployed system exists well before the final week, with the remaining time dedicated to refinement rather than first time integration.

## 2. Data Understanding

This phase establishes a thorough working knowledge of both primary data sources before any cleaning, joining, or modeling begins. Two datasets are examined: the WFP Kenya food prices dataset sourced from HDX, and daily meteorological data retrieved from the NASA POWER API. A third supporting file, mapping market names to geographic coordinates, is also assessed since it is the join key that ties price data to weather data.

The objective of this phase is not only to describe the data, but to surface early evidence for or against the hypotheses formed during Business Understanding, so that the modeling phase begins with informed expectations rather than assumptions.

### 2.1 Data Collection

Two data sources are used, both publicly accessible without authentication, satisfying the capstone's requirement to avoid access barriers.

| Data Source | Access Method | Authentication |
|---|---|---|
| WFP Kenya Food Prices (HDX) | Direct CSV download | None required |
| NASA POWER API | REST API, JSON or CSV response | None required |

The food prices file provides the target variable, historical price observations by market, commodity, and date. The NASA POWER API supplies the exogenous weather variables, specifically rainfall and temperature, used to improve forecast accuracy beyond what price history alone can achieve. A record of the exact extraction date and file version is kept below, since both sources update periodically and reproducibility depends on knowing precisely which snapshot was used.

In [1]:
# import necessary
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import requests
from datetime import datetime

# make wide tables and columns readable within the notebook
pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 150)
sns.set_style("whitegrid")
plt.rcParams["figure.figsize"] = (12, 5)

# Record the extraction timestamp for reproducibility.
extraction_date = datetime.now().strftime("%Y-%m-%d")
print(f"Data extraction date recorded: {extraction_date}")

Data extraction date recorded: 2026-08-22
